In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate
from surprise import accuracy

os.makedirs('../models', exist_ok=True)

## 1. Load filtered data from Day 1

In [3]:
ratings_filtered = pd.read_parquet('../data/ratings_filtered.parquet')
movies_df        = pd.read_csv('../data/movies.csv')

print(f'Ratings: {len(ratings_filtered):,}')
print(f'Users:   {ratings_filtered["userId"].nunique():,}')
print(f'Movies:  {ratings_filtered["movieId"].nunique():,}')

Ratings: 24,644,928
Users:   162,540
Movies:  13,176


In [4]:
from surprise import Reader , Dataset , SVD
from surprise.model_selection import cross_validate

sample = ratings_filtered.sample(n = 500000 , random_state=42)

reader = Reader(rating_scale = (0.5 , 5))

data = Dataset.load_from_df(sample[['userId' , 'movieId' , 'rating']],reader)

algo = SVD(n_factors = 50 , reg_all = 0.02, lr_all = 0.1 , n_epochs = 20 , random_state = 20)

results = cross_validate(algo , data , measures=['MSE' , 'RMSE'],cv = 5 , verbose=True)

Evaluating MSE, RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
MSE (testset)     0.9051  0.9118  0.9170  0.9127  0.9139  0.9121  0.0039  
RMSE (testset)    0.9514  0.9549  0.9576  0.9554  0.9560  0.9550  0.0020  
Fit time          2.21    2.29    2.34    2.30    2.32    2.29    0.04    
Test time         0.38    0.29    0.30    0.30    0.29    0.31    0.04    


In [5]:
print('Training final SVD model on full dataset...')
trainset = data.build_full_trainset()
algo.fit(trainset)
print('Done.')

Training final SVD model on full dataset...
Done.


In [6]:
pred = algo.predict(uid=1, iid=296)
print(f'Predicted rating for user 1 on Pulp Fiction: {pred.est:.2f}')

Predicted rating for user 1 on Pulp Fiction: 3.86


In [7]:
def recommend(user_id , n = 10):
    rated_movies = set(ratings_filtered[ratings_filtered['userId'] == user_id]['movieId'])
    all_movies = set(ratings_filtered['movieId'].unique())
    unseen_movies = all_movies - rated_movies 

    predictions = []
    for movie in unseen_movies:
        pred = algo.predict(uid = user_id , iid = movie)
        predictions.append((movie , pred.est))
    predictions.sort(key = lambda x : x[1] , reverse = True)
    top_n = predictions[:n]

    result = pd.DataFrame(top_n, columns=['movieId', 'predicted_rating'])
    result = result.merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    return result

print(recommend(user_id=1, n=10))

   movieId  predicted_rating                            title  \
0   140737          4.583967             The Lost Room (2006)   
1     2627          4.553247                 Endurance (1999)   
2    26082          4.542931        Harakiri (Seppuku) (1962)   
3    31851          4.526399        Sons of the Desert (1933)   
4     6460          4.512071   Trial, The (Procès, Le) (1962)   
5     3415          4.493283     Mirror, The (Zerkalo) (1975)   
6     2203          4.491997         Shadow of a Doubt (1943)   
7   128620          4.484233                  Victoria (2015)   
8     7013          4.483512  Night of the Hunter, The (1955)   
9     8607          4.483198          Tokyo Godfathers (2003)   

                      genres  
0     Action|Fantasy|Mystery  
1          Documentary|Drama  
2                      Drama  
3                     Comedy  
4                      Drama  
5                      Drama  
6       Crime|Drama|Thriller  
7        Crime|Drama|Romance  
8   D

## 6. Save model

In [ ]:
joblib.dump(algo, '../models/svd_model.pkl')

Saved: ../models/svd_model.pkl
